# Lazypredict

In [2]:
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

In [3]:
# Data Preparation
df = pd.read_excel("Final_PM15-Towel.xlsx")

# Cleaning
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[df['GSM'].str.len() <= 4].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)

# Create Pseudo_Mass Feature
df['pseudo_mass'] = (df['Mean_Stock Flow'] * df['Mean_Stock Consistency'])/ df['Mean_Yankee Speed']

# Create Coating-Release Ratio Feature
df['coating_release_ratio'] = df['Mean_Flow Coating'] / df['Mean_Flow Release']

# X Variables
features = [
    '% NBKP',
    'Mean_Load KWH Tickling Refiner',
    'Mean_Creping',
    'Mean_Jet Wire Ratio',
    'GSM',
    'coating_release_ratio'
]
X = df[features]

# Y Variables

y = df['MDT']

In [4]:
X.tail()

,% NBKP,Mean_Load KWH Tickling Refiner,Mean_Creping,Mean_Jet Wire Ratio,GSM,coating_release_ratio
179,15.016596,324.181316,12.513708,1.015,22.0,0.928052
180,15.016596,325.925372,12.407596,1.015,22.0,0.928047
181,15.016596,328.152200,12.398873,1.015,22.0,0.928053
182,15.016596,325.303419,12.382934,1.015,22.0,0.928050
183,15.016596,326.055321,12.401262,1.015,22.0,0.928054


In [5]:
y.head()

0    1794
1    1977
2    1739
3    1693
4    1686
Name: MDT, dtype: int64

In [6]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [8]:
# Hitung MAPE untuk tiap model
mape_scores = {}

for model_name, y_pred in predictions.items():
    mape = mean_absolute_percentage_error(y_test, y_pred)
    mape_scores[model_name] = mape

# Tambahkan ke tabel hasil
models["MAPE"] = pd.Series(mape_scores)

# Urutkan (semakin kecil semakin baik)
models = models.sort_values(by="MAPE")

In [9]:
print(models)

                               Adjusted R-Squared   R-Squared         RMSE  \
Model                                                                        
Ridge                                    0.884125    0.927578    87.439994   
SGDRegressor                             0.879177    0.924485    89.287315   
BayesianRidge                            0.878870    0.924294    89.400526   
OrthogonalMatchingPursuitCV              0.878792    0.924245    89.429180   
KNeighborsRegressor                      0.867810    0.917381    93.392782   
Lasso                                    0.859759    0.912349    96.194841   
LassoLars                                0.859570    0.912231    96.259788   
PoissonRegressor                         0.846305    0.903941   100.703472   
RidgeCV                                  0.841234    0.900771   102.351153   
LassoCV                                  0.831306    0.894566   105.502837   
PassiveAggressiveRegressor               0.823179    0.889487   

# Regressor

In [11]:
# Import Libraries
import numpy as np

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error)

# Model Dasar
model = ExtraTreesRegressor(random_state=42,n_jobs=1)

# Grid Search Hyper Parameter
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 8, 10, 15],
    'min_samples_split': [2, 4, 6, 10],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 0.5, 0.7],
    'bootstrap': [True, False]
}

# K-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# GGrid Search CV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_percentage_error',
    n_jobs=-1,
    verbose=1
)

# Training + Tuning
grid_search.fit(X, y)

# Model Terbaik
best_model = grid_search.best_estimator_
print("Best Parameters:")
print(grid_search.best_params_)

# Prrediksi
y_pred = best_model.predict(X)

# Matrik Evaluasi
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)
epsilon = 1e-8
mape = np.mean(np.abs((y - y_pred) / (y + epsilon))) * 100

# Hasil
print("\n===== HASIL MODEL TERBAIK =====")
print(f"R2    : {r2:.4f}")
print(f"RMSE  : {rmse:.4f}")
print(f"MAE   : {mae:.4f}")
print(f"MAPE  : {mape:.2f}%")

Fitting 5 folds for each of 1536 candidates, totalling 7680 fits
Best Parameters:
{'bootstrap': False, 'max_depth': 8, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}

===== HASIL MODEL TERBAIK =====
R2    : 0.9268
RMSE  : 92.6493
MAE   : 75.1244
MAPE  : 4.21%


### Pickle Files

In [ ]:
import joblib

In [ ]:
joblib.dump(model, 'model_pm15-towel.pkl')

In [ ]:
features_pkl = X.columns.tolist()
joblib.dump(features_pkl, 'features_pm15-towel.pkl')

### Features Importance

In [ ]:
# Feature Importance
import pandas as pd

feat_imp = pd.DataFrame({
    "Feature": features, 
    "Importance": model.feature_importances_
})

# Urutkan dari terbesar
feat_imp = feat_imp.sort_values(by="Importance", ascending=False)


# Plot
import matplotlib.pyplot as plt
plt.figure()
plt.barh(feat_imp["Feature"], feat_imp["Importance"])
plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Random Forest")

plt.tight_layout()
plt.show()

### SHAP Analysis

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=10)

In [ ]:
# Scatter plot dasar - melihat tren arah secara mendetail
shap.plots.scatter(shap_values[:, "Mean_Load KWH Tickling Refiner"])

In [ ]:
shap.plots.scatter(shap_values[:, "pseudo_mass"])

In [ ]:
shap.plots.scatter(shap_values[:, "% NBKP"])

In [ ]:
shap_values = explainer.shap_values(np.array(X_test))
shap.initjs()
i = 0  # index data

shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    X_test.iloc[i]
)